Creating sampled databases IDS


In [ ]:
import duckdb
import os
import pandas as pd
from pathlib import Path

# ==============================================================================
# 1. SETUP: Define paths and target database sizes
# ==============================================================================

# Path configurations
DATA_DIR = Path('./tpc-h')  # Directory containing TPC-H .tbl files
BASE_SAMPLE_DIR = Path('./tpch-sized-samples')  # Directory for generated samples
GOLD_STANDARD_DIR = Path('./gold_standard_parquet')  # Directory for full Parquet dataset

# Target sizes for sample databases in Megabytes (MB)
TARGET_SIZES_MB = [50, 100, 150, 200, 250]

print(f"Source Data Directory: {DATA_DIR}")
print(f"Output Samples Directory: {BASE_SAMPLE_DIR}")
print(f"Gold Standard Directory: {GOLD_STANDARD_DIR}")
print(f"Target Sample Sizes (MB): {TARGET_SIZES_MB}\n")

# ==============================================================================
# 2. TPC-H SCHEMA DEFINITION
# ==============================================================================
SCHEMAS = {
    'customer': {
        'names': ['C_CUSTKEY', 'C_NAME', 'C_ADDRESS', 'C_NATIONKEY', 'C_PHONE', 
                 'C_ACCTBAL', 'C_MKTSEGMENT', 'C_COMMENT'],
        'dtypes': {
            'C_CUSTKEY': 'INTEGER', 'C_NATIONKEY': 'INTEGER', 
            'C_ACCTBAL': 'DECIMAL(12,2)', 'C_NAME': 'VARCHAR', 
            'C_ADDRESS': 'VARCHAR', 'C_PHONE': 'VARCHAR', 
            'C_MKTSEGMENT': 'VARCHAR', 'C_COMMENT': 'VARCHAR'
        }
    },
    'orders': {
        'names': ['O_ORDERKEY', 'O_CUSTKEY', 'O_ORDERSTATUS', 'O_TOTALPRICE', 
                 'O_ORDERDATE', 'O_ORDERPRIORITY', 'O_CLERK', 'O_SHIPPRIORITY', 'O_COMMENT'],
        'dtypes': {
            'O_ORDERKEY': 'INTEGER', 'O_CUSTKEY': 'INTEGER', 
            'O_TOTALPRICE': 'DECIMAL(12,2)', 'O_ORDERDATE': 'DATE', 
            'O_ORDERSTATUS': 'VARCHAR', 'O_ORDERPRIORITY': 'VARCHAR', 
            'O_CLERK': 'VARCHAR', 'O_SHIPPRIORITY': 'INTEGER', 'O_COMMENT': 'VARCHAR'
        }
    },
    'lineitem': {
        'names': ['L_ORDERKEY', 'L_PARTKEY', 'L_SUPPKEY', 'L_LINENUMBER', 
                 'L_QUANTITY', 'L_EXTENDEDPRICE', 'L_DISCOUNT', 'L_TAX', 
                 'L_RETURNFLAG', 'L_LINESTATUS', 'L_SHIPDATE', 'L_COMMITDATE', 
                 'L_RECEIPTDATE', 'L_SHIPINSTRUCT', 'L_SHIPMODE', 'L_COMMENT'],
        'dtypes': {
            'L_ORDERKEY': 'INTEGER', 'L_PARTKEY': 'INTEGER', 'L_SUPPKEY': 'INTEGER',
            'L_LINENUMBER': 'INTEGER', 'L_QUANTITY': 'DECIMAL(12,2)', 
            'L_EXTENDEDPRICE': 'DECIMAL(12,2)', 'L_DISCOUNT': 'DECIMAL(12,2)', 
            'L_TAX': 'DECIMAL(12,2)', 'L_SHIPDATE': 'DATE', 'L_COMMITDATE': 'DATE',
            'L_RECEIPTDATE': 'DATE', 'L_RETURNFLAG': 'VARCHAR', 
            'L_LINESTATUS': 'VARCHAR', 'L_SHIPINSTRUCT': 'VARCHAR', 
            'L_SHIPMODE': 'VARCHAR', 'L_COMMENT': 'VARCHAR'
        }
    },
    'part': {
        'names': ['P_PARTKEY', 'P_NAME', 'P_MFGR', 'P_BRAND', 'P_TYPE', 
                 'P_SIZE', 'P_CONTAINER', 'P_RETAILPRICE', 'P_COMMENT'],
        'dtypes': {
            'P_PARTKEY': 'INTEGER', 'P_SIZE': 'INTEGER', 
            'P_RETAILPRICE': 'DECIMAL(12,2)', 'P_NAME': 'VARCHAR', 
            'P_MFGR': 'VARCHAR', 'P_BRAND': 'VARCHAR', 'P_TYPE': 'VARCHAR',
            'P_CONTAINER': 'VARCHAR', 'P_COMMENT': 'VARCHAR'
        }
    },
    'supplier': {
        'names': ['S_SUPPKEY', 'S_NAME', 'S_ADDRESS', 'S_NATIONKEY', 
                 'S_PHONE', 'S_ACCTBAL', 'S_COMMENT'],
        'dtypes': {
            'S_SUPPKEY': 'INTEGER', 'S_NATIONKEY': 'INTEGER', 
            'S_ACCTBAL': 'DECIMAL(12,2)', 'S_NAME': 'VARCHAR', 
            'S_ADDRESS': 'VARCHAR', 'S_PHONE': 'VARCHAR', 'S_COMMENT': 'VARCHAR'
        }
    },
    'partsupp': {
        'names': ['PS_PARTKEY', 'PS_SUPPKEY', 'PS_AVAILQTY', 
                 'PS_SUPPLYCOST', 'PS_COMMENT'],
        'dtypes': {
            'PS_PARTKEY': 'INTEGER', 'PS_SUPPKEY': 'INTEGER', 
            'PS_AVAILQTY': 'INTEGER', 'PS_SUPPLYCOST': 'DECIMAL(12,2)', 
            'PS_COMMENT': 'VARCHAR'
        }
    },
    'nation': {
        'names': ['N_NATIONKEY', 'N_NAME', 'N_REGIONKEY', 'N_COMMENT'],
        'dtypes': {
            'N_NATIONKEY': 'INTEGER', 'N_REGIONKEY': 'INTEGER', 
            'N_NAME': 'VARCHAR', 'N_COMMENT': 'VARCHAR'
        }
    },
    'region': {
        'names': ['R_REGIONKEY', 'R_NAME', 'R_COMMENT'],
        'dtypes': {
            'R_REGIONKEY': 'INTEGER', 'R_NAME': 'VARCHAR', 'R_COMMENT': 'VARCHAR'
        }
    }
}

# ==============================================================================
# 3. HELPER FUNCTIONS
# ==============================================================================
def get_total_db_size_bytes(data_dir):
    """Calculates total size of raw .tbl files on disk."""
    total_size = 0
    for table_name in SCHEMAS:
        file_path = data_dir / f"{table_name}.tbl"
        if file_path.exists():
            total_size += file_path.stat().st_size
        else:
            print(f"Warning: File not found at {file_path}")
    return total_size

def preprocess_tbl_file(original_path):
    """
    Reads a .tbl file, removes trailing pipe from each line,
    and writes to a temporary file.
    Returns path to temporary file.
    """
    temp_path = original_path.with_suffix('.tbl.tmp')
    try:
        with original_path.open('r', encoding='utf-8') as infile, \
             temp_path.open('w', encoding='utf-8') as outfile:
            for line in infile:
                cleaned_line = line.rstrip('|\n') + '\n'
                outfile.write(cleaned_line)
        return temp_path
    except Exception as e:
        if temp_path.exists():
            temp_path.unlink()
        raise RuntimeError(f"Error processing {original_path}: {str(e)}")

def read_tbl_statement(file_path, table_name):
    """
    Constructs correct read_csv statement for a TPC-H table.
    """
    schema = SCHEMAS[table_name]
    columns = [f"'{name}' {dtype}" for name, dtype in schema['dtypes'].items()]
    columns_str = ", ".join(columns)
    clean_path = str(file_path).replace('\\', '/')
    return f"read_csv('{clean_path}', delim='|', header=False, columns=[{columns_str}])"

# ==============================================================================
# 4. CREATE GOLD STANDARD (100% CONVERSION)
# ==============================================================================
def create_gold_standard_parquet(data_dir, target_dir):
    """Converts full .tbl dataset to Parquet format."""
    if not data_dir.exists():
        raise FileNotFoundError(f"Source data directory not found at '{data_dir}'")
    
    target_dir.mkdir(parents=True, exist_ok=True)
    con = duckdb.connect(database=':memory:')

    print(f"\n{'='*80}")
    print(f"Creating Gold Standard Parquet Dataset in '{target_dir}'")
    print(f"{'='*80}")
    
    temp_files = []
    try:
        for table_name in SCHEMAS:
            print(f"\nProcessing {table_name}...")
            
            original_path = data_dir / f"{table_name}.tbl"
            if not original_path.exists():
                raise FileNotFoundError(f"Missing table file: {original_path}")
            
            temp_file_path = preprocess_tbl_file(original_path)
            temp_files.append(temp_file_path)

            read_stmt = read_tbl_statement(temp_file_path, table_name)
            parquet_path = target_dir / f"{table_name}.parquet"
            
            print(f"Converting to {parquet_path}...")
            con.execute(f"""
                COPY (SELECT * FROM {read_stmt}) 
                TO '{str(parquet_path).replace('\\', '/')}' (FORMAT PARQUET)
            """)
            
    except Exception as e:
        print(f"\nError creating gold standard: {str(e)}")
        return False
    finally:
        con.close()
        for temp_file in temp_files:
            if temp_file.exists():
                temp_file.unlink()
    
    print("\n✅ Gold standard Parquet dataset created successfully.")
    return True

# ==============================================================================
# 5. CREATE STRATIFIED RELATIONAL SAMPLE
# ==============================================================================
def create_stratified_relational_sample(data_dir, base_sample_dir, target_size_mb):
    """
    Creates relationally-consistent sample using stratified sampling on ORDERS table.
    """
    if not data_dir.exists():
        raise FileNotFoundError(f"Source data directory not found at '{data_dir}'")

    total_db_size_bytes = get_total_db_size_bytes(data_dir)
    if total_db_size_bytes == 0:
        raise ValueError("Could not calculate size of source database.")

    target_size_bytes = target_size_mb * 1024 * 1024
    sampling_fraction = target_size_bytes / total_db_size_bytes

    if not (0 < sampling_fraction <= 1.0001):
        raise ValueError(f"Target size {target_size_mb}MB is not feasible.")

    sample_dir = base_sample_dir / f"tpch-stratified-sample-{target_size_mb}mb"
    sample_dir.mkdir(parents=True, exist_ok=True)
    
    con = duckdb.connect(database=':memory:')
    temp_files = []
    
    try:
        print(f"\n{'='*80}")
        print(f"Creating Stratified Sample for {target_size_mb}MB (fraction: {sampling_fraction:.4f})")
        print(f"{'='*80}")

        # First process all files and store temp paths
        temp_paths = {}
        for table_name in SCHEMAS:
            original_path = data_dir / f"{table_name}.tbl"
            temp_file_path = preprocess_tbl_file(original_path)
            temp_files.append(temp_file_path)
            temp_paths[table_name] = temp_file_path

        # 1. Perform stratified sampling on orders table
        print("\n1. Performing stratified sampling on 'orders' table...")
        orders_read_stmt = read_tbl_statement(temp_paths['orders'], 'orders')
        con.execute(f"CREATE TEMP TABLE orders_full AS SELECT * FROM {orders_read_stmt};")
        
        con.execute(f"""
            CREATE TEMP TABLE sampled_orders AS
            SELECT * FROM orders_full
            USING SAMPLE {sampling_fraction * 100}% (STRATIFY ON O_ORDERSTATUS);
        """)
        
        orders_output = sample_dir / "orders.parquet"
        con.execute(f"COPY sampled_orders TO '{str(orders_output)}' (FORMAT PARQUET);")

        # 2. Propagate sample to related tables
        print("\n2. Propagating sample to related tables...")
        
        # Lineitems for sampled orders
        lineitem_read_stmt = read_tbl_statement(temp_paths['lineitem'], 'lineitem')
        con.execute(f"""
            CREATE TEMP TABLE sampled_lineitem AS
            SELECT li.* FROM {lineitem_read_stmt} AS li
            JOIN sampled_orders so ON li.L_ORDERKEY = so.O_ORDERKEY;
        """)
        lineitem_output = sample_dir / "lineitem.parquet"
        con.execute(f"COPY sampled_lineitem TO '{str(lineitem_output)}' (FORMAT PARQUET);")

        # Customers who made the sampled orders
        customer_read_stmt = read_tbl_statement(temp_paths['customer'], 'customer')
        con.execute(f"""
            CREATE TEMP TABLE sampled_customer AS
            SELECT cust.* FROM {customer_read_stmt} AS cust
            WHERE C_CUSTKEY IN (SELECT DISTINCT O_CUSTKEY FROM sampled_orders);
        """)
        customer_output = sample_dir / "customer.parquet"
        con.execute(f"COPY sampled_customer TO '{str(customer_output)}' (FORMAT PARQUET);")

        # Parts and suppliers related to sampled lineitems
        partsupp_read_stmt = read_tbl_statement(temp_paths['partsupp'], 'partsupp')
        con.execute(f"""
            CREATE TEMP TABLE sampled_partsupp AS
            SELECT ps.* FROM {partsupp_read_stmt} AS ps
            WHERE ps.PS_PARTKEY IN (SELECT DISTINCT L_PARTKEY FROM sampled_lineitem)
            AND ps.PS_SUPPKEY IN (SELECT DISTINCT L_SUPPKEY FROM sampled_lineitem);
        """)
        partsupp_output = sample_dir / "partsupp.parquet"
        con.execute(f"COPY sampled_partsupp TO '{str(partsupp_output)}' (FORMAT PARQUET);")

        # Suppliers for sampled partsupp records
        supplier_read_stmt = read_tbl_statement(temp_paths['supplier'], 'supplier')
        con.execute(f"""
            CREATE TEMP TABLE sampled_supplier AS
            SELECT s.* FROM {supplier_read_stmt} AS s
            WHERE S_SUPPKEY IN (SELECT DISTINCT PS_SUPPKEY FROM sampled_partsupp);
        """)
        supplier_output = sample_dir / "supplier.parquet"
        con.execute(f"COPY sampled_supplier TO '{str(supplier_output)}' (FORMAT PARQUET);")

        # Parts for sampled partsupp records
        part_read_stmt = read_tbl_statement(temp_paths['part'], 'part')
        con.execute(f"""
            CREATE TEMP TABLE sampled_part AS
            SELECT p.* FROM {part_read_stmt} AS p
            WHERE P_PARTKEY IN (SELECT DISTINCT PS_PARTKEY FROM sampled_partsupp);
        """)
        part_output = sample_dir / "part.parquet"
        con.execute(f"COPY sampled_part TO '{str(part_output)}' (FORMAT PARQUET);")

        # Copy nation and region tables as-is (small dimension tables)
        print("\n3. Copying small dimension tables...")
        for table in ['nation', 'region']:
            read_stmt = read_tbl_statement(temp_paths[table], table)
            output_path = sample_dir / f"{table}.parquet"
            con.execute(f"COPY (SELECT * FROM {read_stmt}) TO '{str(output_path)}' (FORMAT PARQUET);")

        print(f"\n✅ Stratified sampling for {target_size_mb}MB complete.")
        print(f"Sample database created at: {sample_dir}")
        return True

    except Exception as e:
        print(f"\nError creating sample: {str(e)}")
        return False
    finally:
        con.close()
        for temp_file in temp_files:
            if temp_file.exists():
                temp_file.unlink()

# ==============================================================================
# MAIN EXECUTION
# ==============================================================================
if __name__ == "__main__":
    try:
        # Create directories if they don't exist
        DATA_DIR.mkdir(exist_ok=True)
        BASE_SAMPLE_DIR.mkdir(exist_ok=True)
        GOLD_STANDARD_DIR.mkdir(exist_ok=True)

        # Step 1: Create Gold Standard Dataset
        gold_success = create_gold_standard_parquet(DATA_DIR, GOLD_STANDARD_DIR)
        
        if not gold_success:
            print("\nStopping process due to error creating gold standard dataset.")
            exit(1)

        # Step 2: Create Sized Samples
        print(f"\n{'='*80}")
        print("Creating Sized Sample Databases using Stratified Sampling")
        print(f"{'='*80}")
        
        for size_mb in TARGET_SIZES_MB:
            print(f"\n{'='*40}")
            print(f"Processing Target Size: {size_mb}MB")
            print(f"{'='*40}")
            
            success = create_stratified_relational_sample(
                data_dir=DATA_DIR,
                base_sample_dir=BASE_SAMPLE_DIR,
                target_size_mb=size_mb
            )
            
            if not success:
                print(f"\nError creating {size_mb}MB sample. Continuing to next size...")
                continue

    except Exception as e:
        print(f"\nFatal error in main execution: {str(e)}")
        exit(1)

    print(f"\n{'='*80}")
    print("All tasks completed successfully.")
    print(f"{'='*80}")

LLM to SQL

In [ ]:
import requests
import json
import duckdb
import time
import csv
import os
import re
from tqdm import tqdm

# ==============================================================================
# 1. SETUP: API, Paths, and Schema
# ==============================================================================

# --- API Setup ---
# Assumes you have a local LLM server running that's compatible with the OpenAI API format.
API_URL = "http://localhost:8000/v1/chat/completions"
HEADERS = {"Content-Type": "application/json"}
MODEL_NAME = "defog/llama-3-sqlcoder-8b" # Or any other SQL-specialized model

# --- Paths ---
BASE_SAMPLE_DIR = './tpch-sized-samples'
# IMPORTANT: You must have a Parquet version of the full dataset for the gold standard comparison.
# You can create this by running the previous sampling script with a target size equal to the full DB size.
GOLD_STANDARD_DIR = "gold_standard_parquet" 

# --- TPC-H Schema (for the AI prompt and query rewriting) ---
TABLE_NAMES = ['customer', 'orders', 'lineitem', 'part', 'supplier', 'partsupp', 'nation', 'region']
SCHEMAS = {
    'customer': ['C_CUSTKEY', 'C_NAME', 'C_ADDRESS', 'C_NATIONKEY', 'C_PHONE', 'C_ACCTBAL', 'C_MKTSEGMENT', 'C_COMMENT'],
    'orders': ['O_ORDERKEY', 'O_CUSTKEY', 'O_ORDERSTATUS', 'O_TOTALPRICE', 'O_ORDERDATE', 'O_ORDERPRIORITY', 'O_CLERK', 'O_SHIPPRIORITY', 'O_COMMENT'],
    'lineitem': ['L_ORDERKEY', 'L_PARTKEY', 'L_SUPPKEY', 'L_LINENUMBER', 'L_QUANTITY', 'L_EXTENDEDPRICE', 'L_DISCOUNT', 'L_TAX', 'L_RETURNFLAG', 'L_LINESTATUS', 'L_SHIPDATE', 'L_COMMITDATE', 'L_RECEIPTDATE', 'L_SHIPINSTRUCT', 'L_SHIPMODE', 'L_COMMENT'],
    'part': ['P_PARTKEY', 'P_NAME', 'P_MFGR', 'P_BRAND', 'P_TYPE', 'P_SIZE', 'P_CONTAINER', 'P_RETAILPRICE', 'P_COMMENT'],
    'supplier': ['S_SUPPKEY', 'S_NAME', 'S_ADDRESS', 'S_NATIONKEY', 'S_PHONE', 'S_ACCTBAL', 'S_COMMENT'],
    'partsupp': ['PS_PARTKEY', 'PS_SUPPKEY', 'PS_AVAILQTY', 'PS_SUPPLYCOST', 'PS_COMMENT'],
    'nation': ['N_NATIONKEY', 'N_NAME', 'N_REGIONKEY', 'N_COMMENT'],
    'region': ['R_REGIONKEY', 'R_NAME', 'R_COMMENT']
}

# --- Global Logs ---
execution_summary = []
nl_sql_log = []

# ==============================================================================
# 2. CORE FUNCTIONS: AI, SQL Execution, and Analysis
# ==============================================================================

def create_schema_prompt():
    """Builds a detailed schema string for the language model prompt."""
    prompt_text = ""
    for table_name, columns in SCHEMAS.items():
        prompt_text += f"Table: {table_name}\nColumns: {', '.join(columns)}\n\n"
    return prompt_text

def natural_language_to_sql(user_command):
    """Sends a user's natural language command to an LLM and returns a SQL query."""
    schema_prompt = create_schema_prompt()
    prompt = f"""
### Task:
Convert the following natural language command into a correctly formatted DuckDB SQL query for the TPC-H schema.

### Notes:
- Use standard SQL functions compatible with DuckDB.
- The database is composed of Parquet files, one for each table.
- Return ONLY the SQL query (no markdown, no explanation).

### Schema:
{schema_prompt}
### User Input:
{user_command}

### SQL Output:
"""
    data = {
        "model": MODEL_NAME,
        "temperature": 0.0,
        "messages": [{"role": "user", "content": prompt}]
    }
    try:
        response = requests.post(API_URL, headers=HEADERS, data=json.dumps(data))
        response.raise_for_status() # Raises an HTTPError for bad responses (4xx or 5xx)
        model_output = response.json()["choices"][0]["message"]["content"]
        
        # Clean up the output to get only the SQL
        sql_match = re.search(r"SELECT.*?;", model_output, re.DOTALL | re.IGNORECASE)
        if sql_match:
            sql = sql_match.group(0).strip()
            nl_sql_log.append([user_command, sql])
            return sql
        else:
            print("⚠️ Warning: Could not extract a valid SELECT query from the model's output.")
            return None
            
    except requests.exceptions.RequestException as e:
        print(f"❌ API request failed: {e}")
        return None
    except (KeyError, IndexError) as e:
        print(f"❌ Failed to parse API response: {e}")
        return None

def rewrite_sql_for_duckdb(sql_query, db_path):
    """Dynamically rewrites a standard SQL query to query Parquet files."""
    for table in TABLE_NAMES:
        pattern = r'\b' + table + r'\b'
        file_path = os.path.join(db_path, f"{table}.parquet").replace('\\', '/')
        replacement = f"read_parquet('{file_path}')"
        sql_query = re.sub(pattern, replacement, sql_query, flags=re.IGNORECASE)
    return sql_query

def execute_sql(db_path, sql_query):
    """Executes a SQL query against a DuckDB database (directory of Parquet files)."""
    conn = duckdb.connect(database=':memory:')
    rewritten_sql = rewrite_sql_for_duckdb(sql_query, db_path)
    
    start = time.time()
    result = conn.execute(rewritten_sql).fetchall()
    end = time.time()
    conn.close()
    return result, end - start

# (Helper functions: aggregate_results, compute_relative_error remain the same)
def aggregate_results(results):
    if not results or not results[0]: return None
    try: return sum(row[0] for row in results if isinstance(row[0], (int, float)))
    except (TypeError, IndexError): return None

def compute_relative_error(gold, sample):
    if gold is None or sample is None or gold == 0: return None
    try: return abs(sample - gold) / gold * 100
    except (TypeError, ZeroDivisionError): return None

# ==============================================================================
# 3. EXPERIMENT EXECUTION
# ==============================================================================

def run_experiment(sql_query, gold_db_path, sample_db_paths):
    """Runs a query on the full DB and all sample DBs, then compares results."""
    print("\n🔍 Executing on full database (Gold Standard)...")
    try:
        gold_results, gold_time = execute_sql(gold_db_path, sql_query)
        gold_aggregated = aggregate_results(gold_results)
        print(f"✅ Full DB Result: {gold_aggregated} | Execution Time: {gold_time:.4f} seconds")
    except Exception as e:
        print(f"❌ Failed to execute on gold standard DB: {e}")
        return

    contains_avg = "AVG" in sql_query.upper()
    if contains_avg:
        print("⚠️ Query contains 'AVG'. Results for AVG will not be scaled.")

    gold_db_size = get_directory_size(gold_db_path)

    for db_path in tqdm(sample_db_paths, desc="🔁 Processing Sample DBs"):
        sample_db_size = get_directory_size(db_path)
        if gold_db_size == 0 or sample_db_size == 0:
            continue
        inverse_scaling = gold_db_size / sample_db_size

        try:
            sample_results, sample_time = execute_sql(db_path, sql_query)
            raw_result = aggregate_results(sample_results)
            scaled_result = raw_result if contains_avg else (raw_result * inverse_scaling if raw_result is not None else None)
            rel_error = compute_relative_error(gold_aggregated, scaled_result)
            db_name = os.path.basename(db_path)

            execution_summary.append([
                sql_query, db_name, raw_result, scaled_result,
                f"{sample_time:.4f}", f"{gold_time:.4f}",
                f"{rel_error:.2f}" if rel_error is not None else "N/A",
                f"{sample_time/gold_time*100:.2f}" if gold_time > 0 else "N/A"
            ])
        except Exception as e:
            print(f"❌ Failed on {db_path}: {e}")
            execution_summary.append([sql_query, os.path.basename(db_path), None, None, "Error", f"{gold_time:.4f}", "Error", "Error"])

def get_directory_size(directory_path):
    total_size = 0
    for dirpath, _, filenames in os.walk(directory_path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            if not os.path.islink(fp):
                total_size += os.path.getsize(fp)
    return total_size

# ==============================================================================
# 4. EXPORTING AND MAIN LOOP
# ==============================================================================

def export_results():
    if not execution_summary:
        print("\nNo results to export.")
        return
    # Export 1: Full details
    with open("experiment_results.csv", "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["Query", "Database", "Raw Sample Result", "Scaled Result", "Execution Time (s)", "Gold Execution Time (s)", "Relative Error (%)", "Speedup vs Gold (%)"])
        writer.writerows(execution_summary)
    # Export 2: Error and Overhead
    with open("errors_overhead.csv", "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["Query", "Database", "Relative Error (%)", "Speedup vs Gold (%)"])
        for row in execution_summary:
            writer.writerow([row[0], row[1], row[6], row[7]])
    # Export 3: NL to SQL log
    with open("queries_log.txt", "w") as f:
        for nl, sql in nl_sql_log:
            f.write(f"Natural Language: {nl}\nSQL Query: {sql}\n\n")

    print("\n✅ Saved:")
    print(" - experiment_results.csv (full details)")
    print(" - errors_overhead.csv (error + speedup)")
    print(" - queries_log.txt (natural language input + SQL output)")

def main():
    if not os.path.exists(GOLD_STANDARD_DIR):
        print(f"❌ Error: Gold standard directory '{GOLD_STANDARD_DIR}' not found.")
        print("Please create a Parquet version of the full TPC-H dataset in that directory before running experiments.")
        return

    sample_db_paths = sorted([os.path.join(BASE_SAMPLE_DIR, d) for d in os.listdir(BASE_SAMPLE_DIR) if os.path.isdir(os.path.join(BASE_SAMPLE_DIR, d))])
    
    while True:
        user_input = input("\n🧠 Enter your query in plain English (or type 'exit'): ")
        if user_input.lower() == "exit":
            break

        sql_query = natural_language_to_sql(user_input)
        if sql_query:
            print(f"\n📝 Generated SQL Query:\n{sql_query}")
            run_experiment(sql_query, GOLD_STANDARD_DIR, sample_db_paths)
        else:
            print("Could not generate SQL query. Please try another command.")

    export_results()

if __name__ == "__main__":
    main()


SQL

In [ ]:
import duckdb
import time
import csv
import os
import re
from tqdm import tqdm

# Global list to store results for final export
execution_summary = []

# --- TPC-H Schema and Paths ---
# This helps the script find tables and calculate sizes accurately.
BASE_SAMPLE_DIR = './tpch-sized-samples' # Path to the generated samples
GOLD_STANDARD_DIR = './gold_standard_parquet' # Path to the full Parquet dataset

TABLE_NAMES = [
    'customer', 'orders', 'lineitem', 'part', 
    'supplier', 'partsupp', 'nation', 'region'
]

def get_directory_size(directory_path):
    """Calculates the total size of all files in a directory."""
    total_size = 0
    for dirpath, _, filenames in os.walk(directory_path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            if not os.path.islink(fp):
                total_size += os.path.getsize(fp)
    return total_size

def rewrite_sql_for_duckdb(sql_query, db_path):
    """
    Dynamically rewrites a standard SQL query to work with DuckDB's 
    read_parquet function, adding necessary table aliases.
    """
    for table in TABLE_NAMES:
        # Explicitly match 'FROM table' and 'JOIN table' to avoid incorrectly
        # replacing column names that are the same as table names.
        from_pattern = re.compile(r'\bFROM\s+' + table + r'\b', re.IGNORECASE)
        join_pattern = re.compile(r'\bJOIN\s+' + table + r'\b', re.IGNORECASE)

        file_path = os.path.join(db_path, f"{table}.parquet").replace('\\', '/')
        
        # Define the replacements, which include reading from Parquet and adding an alias
        from_replacement = f"FROM read_parquet('{file_path}') AS {table}"
        join_replacement = f"JOIN read_parquet('{file_path}') AS {table}"
        
        # Apply the replacements
        sql_query = from_pattern.sub(from_replacement, sql_query)
        sql_query = join_pattern.sub(join_replacement, sql_query)
        
    return sql_query

def execute_sql(db_path, sql_query):
    """Executes a SQL query against a DuckDB database (represented by a directory of files)."""
    conn = duckdb.connect(database=':memory:')
    rewritten_sql = rewrite_sql_for_duckdb(sql_query, db_path)
    
    start = time.time()
    result = conn.execute(rewritten_sql).fetchall()
    end = time.time()
    conn.close()
    return result, end - start

def aggregate_results(results, sql_query):
    """
    Aggregates query results into a single number for comparison.
    Handles simple aggregations and GROUP BY queries intelligently.
    """
    if not results or not results[0]:
        return None

    is_group_by = "GROUP BY" in sql_query.upper()
    is_avg = "AVG" in sql_query.upper()

    try:
        aggregated_value = None
        # For GROUP BY queries, aggregate the result column (usually the second one)
        if is_group_by:
            result_column_index = 1 if len(results[0]) > 1 else 0
            valid_rows = [row[result_column_index] for row in results if row[result_column_index] is not None]
            
            if not valid_rows:
                return None

            if is_avg:
                # Calculate the average of the averages
                aggregated_value = sum(valid_rows) / len(valid_rows)
            else:
                # For SUM() or COUNT(), calculate the sum of the sums/counts
                aggregated_value = sum(valid_rows)
        # For simple queries without GROUP BY, return the single value
        else:
            aggregated_value = results[0][0]
        
        # FIX: Convert the final aggregated value (which may be a Decimal) to a float
        # to prevent TypeErrors in subsequent calculations (e.g., multiplication with a float).
        return float(aggregated_value) if aggregated_value is not None else None
            
    except (TypeError, IndexError, ZeroDivisionError):
        return None


def compute_relative_error(gold, sample):
    """Computes the relative error between a sample result and the gold standard."""
    if gold is None or sample is None or gold == 0:
        return None
    try:
        return abs(sample - gold) / abs(gold) * 100
    except (TypeError, ZeroDivisionError):
        return None

def run_experiment(sql_query, gold_db_path, sample_db_paths):
    """Runs a query on the full DB and all sample DBs, then compares results."""
    print("\n🔍 Executing on full database (Gold Standard)...")
    
    try:
        gold_results, gold_time = execute_sql(gold_db_path, sql_query)
        gold_aggregated = aggregate_results(gold_results, sql_query)
        if gold_aggregated is None:
            print("❌ Gold standard query returned no result.")
            return
        print(f"✅ Full DB Aggregated Result: {gold_aggregated:.4f} | Execution Time: {gold_time:.4f} seconds")
    except Exception as e:
        print(f"❌ Failed to execute on gold standard DB: {e}")
        return

    contains_avg = "AVG" in sql_query.upper()
    if contains_avg:
        print("⚠️  Query contains 'AVG'. Results for AVG are not scaled.")

    gold_db_size = get_directory_size(gold_db_path)

    for db_path in tqdm(sample_db_paths, desc="🔁 Processing Sample DBs"):
        sample_db_size = get_directory_size(db_path)
        
        if gold_db_size == 0 or sample_db_size == 0:
            print(f"Skipping {db_path} due to zero size.")
            continue
            
        inverse_scaling = gold_db_size / sample_db_size

        try:
            sample_results, sample_time = execute_sql(db_path, sql_query)
            raw_result = aggregate_results(sample_results, sql_query)

            scaled_result = None
            if raw_result is not None:
                # AVG is self-normalizing, so we don't scale it.
                # SUM, COUNT, etc., must be scaled up.
                scaled_result = raw_result if contains_avg else raw_result * inverse_scaling

            rel_error = compute_relative_error(gold_aggregated, scaled_result)

            db_name = os.path.basename(db_path)
            
            # Append results to the global summary list
            execution_summary.append([
                sql_query, db_name, 
                f"{raw_result:.4f}" if raw_result is not None else "N/A",
                f"{scaled_result:.4f}" if scaled_result is not None else "N/A",
                f"{sample_time:.4f}", f"{gold_time:.4f}",
                f"{rel_error:.2f}" if rel_error is not None else "N/A",
                f"{sample_time/gold_time*100:.2f}" if gold_time > 0 else "N/A"
            ])

        except Exception as e:
            print(f"❌ Failed on {db_path}: {e}")
            execution_summary.append([sql_query, os.path.basename(db_path), None, None, None, f"{gold_time:.4f}", "Error", "Error"])

def export_results():
    """Exports the summary of all experiments to CSV files."""
    if not execution_summary:
        print("\nNo results to export.")
        return

    # Export 1: Full details
    with open("experiment_results.csv", "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "Query", "Database", "Raw Sample Result", "Scaled Result",
            "Execution Time (s)", "Gold Execution Time (s)",
            "Relative Error (%)", "Speedup vs Gold (%)"
        ])
        writer.writerows(execution_summary)

    # Export 2: Error and Overhead only
    with open("errors_overhead.csv", "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["Query", "Database", "Relative Error (%)", "Speedup vs Gold (%)"])
        for row in execution_summary:
            writer.writerow([row[0], row[1], row[6], row[7]])

    print("\n✅ Results exported to:")
    print(" - experiment_results.csv (full details)")
    print(" - errors_overhead.csv (relative error + speedup only)")

def main():
    """Main loop to run the experiment."""
    if not os.path.exists(GOLD_STANDARD_DIR):
        print(f"Error: Gold standard directory '{GOLD_STANDARD_DIR}' not found.")
        print("Please run the data sampling script first to create the full Parquet dataset.")
        return
        
    sample_db_paths = [os.path.join(BASE_SAMPLE_DIR, d) for d in sorted(os.listdir(BASE_SAMPLE_DIR)) if os.path.isdir(os.path.join(BASE_SAMPLE_DIR, d))]

    while True:
        user_input = input("\n🧠 Enter your SQL query (e.g., SELECT SUM(O_TOTALPRICE) FROM orders) or type 'exit': ")
        if user_input.lower() == "exit":
            break

        if not user_input.strip().upper().startswith("SELECT"):
            print("⚠️ Only SELECT queries are allowed.")
            continue

        sql_query = user_input.strip()
        run_experiment(sql_query, GOLD_STANDARD_DIR, sample_db_paths)

    export_results()

if __name__ == "__main__":
    main()
